# 31 — Kubernetes & EKS Production Mastery

**Role:** Senior AWS DevOps Engineer | **Focus:** EKS, Deployments, Scaling, Production Troubleshooting

---

## Section A: EKS Architecture & Setup

### Q1: EKS Cluster Design
**Question:** Walk me through how you'd set up a production EKS cluster from scratch. What decisions do you make regarding networking, node groups, and add-ons?

**Expected Answer:**
- **Networking**: Private API endpoint, private subnets for nodes, VPC CNI for pod networking.
- **Node Groups**: Managed node groups (easier patching) vs. self-managed (more control). Separate node groups for system vs. application workloads.
- **Instance types**: Mixed instance types in node group for Spot diversity. Graviton for cost savings.
- **Add-ons**: CoreDNS, kube-proxy, VPC CNI (managed add-ons), AWS LB Controller, EBS CSI driver.
- **IRSA**: IAM Roles for Service Accounts for all workloads needing AWS access.
- **Logging**: Control plane logs to CloudWatch (API server, audit, authenticator).

---

### Q2: EKS vs. Self-Managed K8s
**Question:** Why EKS instead of running your own Kubernetes with `kubeadm` on EC2?

**Expected Answer:**
- **Managed control plane**: AWS handles etcd, API server HA, version upgrades.
- **Integrations**: Native ALB Ingress, IRSA, CloudWatch Container Insights.
- **Security**: OIDC identity provider, envelope encryption for secrets with KMS.
- **Trade-off**: Less control over control plane config; $0.10/hr per cluster cost.
- **When self-managed**: Edge cases, air-gapped environments, or specific K8s version needs.

## Section B: Deployments & Scaling

### Q3: Deployment Troubleshooting
**Question:** A deployment is stuck in `Pending` state. Walk me through your debugging process.

**Expected Answer:**
```bash
# Step 1: Check pod status
kubectl describe pod <pod-name> -n <namespace>

# Common causes:
# - Insufficient resources: "0/3 nodes are available: insufficient cpu/memory"
# - Node selector/affinity mismatch
# - PVC not bound (storage class issues)
# - Image pull errors (wrong tag, ECR auth expired)
# - Taints without tolerations

# Step 2: Check events
kubectl get events --sort-by=.metadata.creationTimestamp -n <namespace>

# Step 3: Check node capacity
kubectl describe nodes | grep -A5 'Allocated resources'
```

---

### Q4: HPA vs. VPA vs. Karpenter
**Question:** Explain the difference between HPA, VPA, and Karpenter. Can they work together?

**Expected Answer:**
- **HPA (Horizontal Pod Autoscaler)**: Scales pod count based on CPU/memory/custom metrics.
- **VPA (Vertical Pod Autoscaler)**: Adjusts pod resource requests/limits. Cannot run with HPA on the same metric.
- **Karpenter**: Node-level autoscaler. Provisions right-sized nodes for pending pods. Replaces Cluster Autoscaler.
- **Together**: HPA scales pods → pods go `Pending` → Karpenter provisions new nodes. VPA can run in `recommend` mode alongside HPA.
- **Custom metrics HPA**: Scale on SQS queue depth, request latency, etc. via KEDA or Prometheus Adapter.

---

### Q5: Resource Requests & Limits
**Question:** Should you always set both `requests` and `limits` for CPU and memory? What happens when you don't?

**Expected Answer:**
- **Requests**: Used by scheduler for placement. Always set.
- **Memory limits**: Set to prevent OOMKill of other pods. Container is killed if it exceeds.
- **CPU limits**: Controversial. CPU is compressible — pods get throttled, not killed. Many teams now REMOVE CPU limits to avoid throttling.
- **QoS Classes**: `Guaranteed` (requests == limits), `Burstable` (requests < limits), `BestEffort` (no requests). Eviction order: BestEffort first.
- **LimitRange**: Enforce defaults at namespace level so no pod runs without requests.

## Section C: Cluster Upgrades & Maintenance

### Q6: EKS Upgrade Strategy
**Question:** How do you upgrade an EKS cluster from 1.28 to 1.30 in production with zero downtime?

**Expected Answer:**
- **Step 1**: Check deprecated APIs with `pluto` or `kubent`. Fix manifests.
- **Step 2**: Upgrade control plane (1.28 → 1.29 → 1.30). One minor version at a time.
- **Step 3**: Upgrade managed add-ons (CoreDNS, kube-proxy, VPC CNI) to compatible versions.
- **Step 4**: Rolling upgrade node groups. Create new node group with 1.30 AMI, drain old nodes.
- **PDBs**: Ensure PodDisruptionBudgets allow at most 1 unavailable.
- **Rollback**: You CANNOT downgrade the EKS control plane. Test thoroughly in staging.

---

### Q7: Helm vs. Kustomize
**Question:** When do you choose Helm over Kustomize for K8s manifest management?

**Expected Answer:**
- **Helm**: Templating engine, package manager, release management, rollback. Best for third-party charts (nginx-ingress, cert-manager).
- **Kustomize**: Patch-based, no templating, native `kubectl` support. Best for internal apps with env-specific overlays.
- **Together**: Use Helm for community charts, Kustomize for your own apps. Or `helmfile` for declarative Helm.
- **GitOps**: ArgoCD supports both; renders Helm templates server-side.

---

### Q8: Network Policies & Service Mesh
**Question:** How do you implement zero-trust networking within an EKS cluster?

**Expected Answer:**
- **Network Policies**: Default-deny all ingress/egress per namespace. Explicitly allow needed flows.
- **CNI requirement**: VPC CNI alone doesn't enforce Network Policies. Need Calico or Cilium as policy engine.
- **Service Mesh (Istio/Linkerd)**: mTLS between all pods, fine-grained AuthorizationPolicies.
- **AWS App Mesh**: AWS-native alternative, integrates with Envoy sidecars.
- **DNS policies**: Restrict egress DNS to only internal domains for sensitive workloads.

## Section D: Production Scenarios

### Q9: CrashLoopBackOff Deep Dive
**Question:** A pod is in `CrashLoopBackOff`. You cannot `exec` into it because it dies instantly. How do you debug?

**Expected Answer:**
```bash
# Check logs from the last crash
kubectl logs <pod> --previous

# Check events for OOMKilled
kubectl describe pod <pod> | grep -i oom

# Override the entrypoint to keep it alive
kubectl debug <pod> -it --copy-to=debug-pod --container=app -- /bin/sh
# or edit deployment: command: [\"sleep\", \"infinity\"]

# Common causes:
# - Missing env vars or config maps
# - DB connection refused (wrong endpoint or SG)
# - OOMKilled (memory limit too low)
# - Readiness probe failing immediately
```

---

### Q10: Multi-Tenant Cluster Strategy
**Question:** Multiple teams share one EKS cluster. How do you ensure isolation?

**Expected Answer:**
- **Namespaces**: One per team/service.
- **ResourceQuotas**: Limit CPU/memory per namespace.
- **LimitRanges**: Default requests/limits for pods.
- **Network Policies**: Namespace-to-namespace isolation.
- **RBAC**: Team-scoped ClusterRoles bound to their namespace.
- **Priority Classes**: Ensure critical workloads aren't preempted by lower-priority ones.